# Part 1 - Read, validate, clean and enrich


In [ ]:
from pathlib import Path
import csv
import json
import numbers
import numpy as np
import pandas as pd
import requests

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

## 1.1 - Read with csv, convert types, keep rejected original rows

In [ ]:
def parse_row(row):
    converted = row.copy()
    for column in ['age', 'fee_inr', 'prior_no_shows']:
        converted[column] = int(row[column])
    converted['distance_km'] = float(row['distance_km'])
    return converted


def read_foundations(path):
    accepted, rejected = [], []
    with open(path, newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        print('Header:', reader.fieldnames)
        for row in reader:
            try:
                accepted.append(parse_row(row))
            except ValueError:
                rejected.append(row.copy())
    print('Read:', len(accepted) + len(rejected))
    print('Accepted:', len(accepted), '| Rejected:', len(rejected))
    with open('rejected_rows.json', 'w', encoding='utf-8') as handle:
        json.dump(rejected, handle, indent=2, ensure_ascii=False)
    return accepted, rejected

if __name__ == '__main__':
    accepted, rejected = read_foundations('appointments.csv')

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Read: 1020
Accepted: 988 | Rejected: 32


## 1.2-1.4 - Pandas cleaning, derived features and NumPy calculations
Follow the brief's operation order. Medians/fences use all rows here as required
by the assignment; this is a disclosed limitation, not a production ML pipeline.

In [ ]:
COLUMNS = ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
           'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
           'visit_type', 'reminder_sent', 'prior_no_shows', 'status',
           'lead_days', 'no_show']
CLINIC_COLUMNS = ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude',
                  'longitude', 'temp_c']


def clean_appointments(path):
    df = pd.read_csv(path)
    print('Raw shape:', df.shape)
    df.info()
    print('Missing values:\n', df.isnull().sum())
    print('Exact duplicates:', df.duplicated().sum())
    raw_count = len(df)
    df = df.drop_duplicates().copy()
    print('After dropping duplicates:', len(df))
    print('Raw channel spellings:', df['booking_channel'].nunique())
    df['booking_channel'] = df['booking_channel'].str.strip().str.title()
    print('Clean channel labels:', sorted(df['booking_channel'].unique()))
    print('After channel cleaning:', len(df))
    # The supplied files use ISO dates. Do not guess ambiguous date formats.
    df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
    df['booked_on'] = pd.to_datetime(df['booked_on'])
    print('After date conversion:', len(df))
    df = df.dropna(subset=['appt_date', 'status']).copy()
    print('After dropping unusable dates/status:', len(df))
    invalid_age = (df['age'] < 1) | (df['age'] > 100)
    print('Invalid ages:', int(invalid_age.sum()))
    df.loc[invalid_age, 'age'] = np.nan
    print('Missing ages after validity check:', int(df['age'].isna().sum()))
    age_median = df['age'].median()
    distance_median = df['distance_km'].median()
    print('Age median:', age_median, '| Distance median:', distance_median)
    df['age'] = df['age'].fillna(age_median).round().astype(int)
    df['distance_km'] = df['distance_km'].fillna(distance_median)
    capped = {}
    for column in ['distance_km', 'fee_inr']:
        q1, q3 = df[column].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        capped[column] = int(((df[column] < lower) | (df[column] > upper)).sum())
        print(column, {'Q1': q1, 'Q3': q3, 'IQR': iqr, 'lower': lower,
                       'upper': upper, 'capped': capped[column]})
        df[column] = np.clip(df[column], lower, upper)
    df['distance_km'] = df['distance_km'].round(1)
    df['fee_inr'] = df['fee_inr'].round().astype(int)
    assert set(df['status']) <= {'Attended', 'No-Show'}, 'Unknown status'
    df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
    df['no_show'] = df['status'].eq('No-Show').astype(int)
    assert df['lead_days'].ge(0).all(), 'Appointment before booking'
    X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()
    means = X.mean(axis=0)
    mask = X[:, 3] > 14
    distance = X[:, 1]
    sd = np.std(distance)  # Population standard deviation: ddof=0.
    z_scores = (distance - np.mean(distance)) / sd if sd else np.zeros_like(distance)
    print('Means [age, distance, fee, lead_days]:', means)
    print('Lead days > 14:', mask.sum(), f'({100 * mask.mean():.1f}%)')
    print('Distance |z| > 2:', int((np.abs(z_scores) > 2).sum()))
    print('Lead-day range:', df['lead_days'].min(), df['lead_days'].max())
    print('No-shows:', df['no_show'].sum(), f'({100 * df.no_show.mean():.1f}%)')
    summary = {'raw_rows': raw_count, 'clean_rows': len(df),
               'no_shows': int(df.no_show.sum()), 'no_show_pct': 100 * df.no_show.mean(),
               'means': means.tolist(), 'lead_gt14': int(mask.sum()),
               'distance_z_gt2': int((np.abs(z_scores) > 2).sum()), 'capped': capped}
    return df, summary

if __name__ == '__main__':
    df, cleaning_summary = clean_appointments('appointments.csv')

Raw shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
Missing values:
 appointment_id      0
patient_id          0
clinic_id

## 1.5 - Live clinic enrichment with one retry and honest missing values
This is today's forecast's first hourly value, not weather for the 2025 bookings.
It is never included as a prediction feature.

In [ ]:
def fetch_temperature(clinic, get=requests.get):
    params = {'latitude': float(clinic['latitude']),
              'longitude': float(clinic['longitude']),
              'hourly': 'temperature_2m', 'forecast_days': 1, 'timezone': 'UTC'}
    for attempt in range(2):
        try:
            response = get('https://api.open-meteo.com/v1/forecast',
                           params=params, timeout=10)
            if response.status_code != 200:
                raise ValueError(f'HTTP {response.status_code}')
            values = response.json().get('hourly', {}).get('temperature_2m')
            if not values or values[0] is None:
                raise ValueError('Missing first hourly temperature')
            temp = float(values[0])
            if not np.isfinite(temp):
                raise ValueError('Non-finite temperature')
            print(clinic['clinic_name'], 'status=200', 'temp_c=', temp)
            return temp
        except (requests.exceptions.RequestException, ValueError, TypeError, AttributeError) as exc:
            print(clinic['clinic_name'], f'attempt {attempt + 1}/2:', str(exc))
    return None


def enrich_clinics(path):
    clinics = pd.read_csv(path)
    assert clinics['clinic_id'].is_unique
    clinics['temp_c'] = [fetch_temperature(row) for _, row in clinics.iterrows()]
    clinics = clinics[CLINIC_COLUMNS]
    clinics.to_csv('clinics_enriched.csv', index=False)
    return clinics

if __name__ == '__main__':
    clinics = enrich_clinics('clinics.csv')

Kothrud Family Clinic status=200 temp_c= 17.4
Hadapsar Care Centre status=200 temp_c= 18.7
Sitabuldi Health Point status=200 temp_c= 19.9
Malviya Nagar Clinic status=200 temp_c= 19.7
Vaishali Nagar Clinic status=200 temp_c= 19.7
Edappally Medical Centre status=200 temp_c= 25.7
Vijay Nagar Clinic status=200 temp_c= 18.3
Gomti Nagar Care Point status=200 temp_c= 18.7


## 1.6 - Export CSV and SQL, with integrity checks
Clinics are inserted first. Missing temperatures become unquoted SQL NULL.
SQL text is only for this fixed dataset; use parameterised queries in applications.

In [ ]:
def sql_literal(value):
    if pd.isna(value):
        return 'NULL'
    if isinstance(value, numbers.Integral):
        return str(int(value))
    if isinstance(value, numbers.Real):
        return repr(float(value))
    # Reject backslashes for consistent MySQL/SQLite literal semantics.
    text = str(value)
    if '\\' in text:
        raise ValueError('Backslash in SQL text: use a parameterised loader')
    return "'" + text.replace("'", "''") + "'"


def export_files(df, clinics, summary):
    clean = df.sort_values('appointment_id').copy()[COLUMNS]
    for column in ['booked_on', 'appt_date']:
        clean[column] = clean[column].dt.strftime('%Y-%m-%d')
    print('appointment_id unique:', clean.appointment_id.is_unique)
    print('Missing cells:', int(clean.isna().sum().sum()))
    assert clean.appointment_id.is_unique
    assert not clean.isna().any().any()
    assert set(clean.clinic_id) <= set(clinics.clinic_id)
    clean.to_csv('appointments_clean.csv', index=False)
    with open('load_data.sql', 'w', encoding='utf-8') as handle:
        for table_name, frame in [('clinics', clinics), ('appointments', clean)]:
            for row in frame.itertuples(index=False, name=None):
                values = ','.join(sql_literal(value) for value in row)
                handle.write(f'INSERT INTO {table_name} VALUES ({values});\n')
    (RESULTS / 'part1_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print('INSERT statements:', len(clinics) + len(clean))
    # These are checks against the supplied assignment, never substitute results.
    observed = [summary['raw_rows'], len(clean), int(clean.no_show.sum()),
                summary['lead_gt14'], summary['distance_z_gt2']]
    assert observed == [1020, 990, 218, 296, 53], f'Brief mismatch: {observed}'
    assert np.allclose(summary['means'], [48.17, 9.50, 635.71, 11.82], atol=0.01)
    return clean

if __name__ == '__main__':
    clean = export_files(df, clinics, cleaning_summary)
    assert (len(accepted), len(rejected)) == (988, 32)
    print('Part 1 complete.')

appointment_id unique: True
Missing cells: 0
INSERT statements: 998
Part 1 complete.


In [ ]:
saved = pd.read_csv("appointments_clean.csv")
enriched = pd.read_csv("clinics_enriched.csv")

print("Clean CSV shape:", saved.shape)                  # (990, 16)
print("Distinct appointment IDs:", saved.appointment_id.nunique())  # 990
print("Missing cells:", saved.isna().sum().sum())         # 0
print("Largest cleaned fee:", saved.fee_inr.max())        # 1219
print("Clinic rows:", len(enriched))                     # 8
sql = Path("load_data.sql").read_text(encoding="utf-8")
print("Actual INSERT statements:",
      sum(line.startswith("INSERT INTO ") for line in sql.splitlines()))

Clean CSV shape: (990, 16)
Distinct appointment IDs: 990
Missing cells: 0
Largest cleaned fee: 1219
Clinic rows: 8
Actual INSERT statements: 998


In [ ]:
import pandas as pd
clean_add = pd.read_csv("/content/appointments_clean.csv")

In [ ]:
clean_add.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   990 non-null    object 
 1   patient_id       990 non-null    object 
 2   clinic_id        990 non-null    object 
 3   booked_on        990 non-null    object 
 4   appt_date        990 non-null    object 
 5   age              990 non-null    int64  
 6   gender           990 non-null    object 
 7   distance_km      990 non-null    float64
 8   fee_inr          990 non-null    int64  
 9   booking_channel  990 non-null    object 
 10  visit_type       990 non-null    object 
 11  reminder_sent    990 non-null    object 
 12  prior_no_shows   990 non-null    int64  
 13  status           990 non-null    object 
 14  lead_days        990 non-null    int64  
 15  no_show          990 non-null    int64  
dtypes: float64(1), int64(5), object(10)
memory usage: 123.9+ KB


In [ ]:
clean_add['no_show'].value_counts()

,count
no_show,
0,772
1,218
